# Notebook 00 — Did the fine-tune actually work?

**Workshop:** Agentic AI for Scientists — Week 5 (Evaluation & Benchmarking)
**Block:** structured evaluation (25 min)
**Goal:** Take the models Week 4 produced and **measure** them against held-out ground truth. A falling training loss told us the model was *learning something*; it did **not** tell us the model is *good*. This notebook turns "is it good?" into numbers.

**The trap we're avoiding:** judging a fine-tune by reading three cherry-picked outputs. That's how teams ship models that look great in the demo and fail in production. We evaluate on a held-out **test split the model never saw**, with metrics that match what we actually care about.

**What we measure** (our Week 4 schema makes this clean):
- **JSON validity** — can we even parse the output? (a format-compliance metric)
- **Disease accuracy** — did it name the right condition?
- **Symptom / treatment F1** — set overlap between predicted and gold lists
- **Answer-summary ROUGE-L** — text overlap for the free-text field

> Because Week 4 made the answer a *structured object*, evaluation is field-by-field and honest. This is the payoff of "schema design is eval design."

## 0. Setup

In [ ]:
%pip install -q unsloth rouge-score
%pip install -q --no-deps transformers

In [ ]:
# A small held-out test set (same schema as Week 4's data/medquad_*.jsonl).
# If you have Week 4's data/medquad_test.jsonl, we use that instead.
import json
from pathlib import Path

INLINE_TEST = [
 {"question": "What are the symptoms of Hashimoto's disease?",
  "gold": {"disease": "Hashimoto's disease", "symptoms": ["fatigue", "weight gain", "cold intolerance", "constipation", "dry skin"], "treatment": ["levothyroxine"]}},
 {"question": "How is type 2 diabetes treated?",
  "gold": {"disease": "Type 2 diabetes", "symptoms": ["increased thirst", "frequent urination", "fatigue", "blurred vision"], "treatment": ["metformin", "lifestyle changes", "insulin"]}},
 {"question": "What are the warning signs of a stroke?",
  "gold": {"disease": "Stroke", "symptoms": ["facial drooping", "arm weakness", "slurred speech", "sudden confusion", "vision loss"], "treatment": ["thrombolysis", "thrombectomy", "aspirin"]}},
 {"question": "What are the symptoms of anemia?",
  "gold": {"disease": "Iron deficiency anemia", "symptoms": ["fatigue", "pallor", "shortness of breath", "dizziness"], "treatment": ["iron supplements", "dietary iron"]}},
 {"question": "What causes asthma?",
  "gold": {"disease": "Asthma", "symptoms": ["wheezing", "cough", "chest tightness", "shortness of breath"], "treatment": ["inhaled corticosteroids", "beta agonists"]}},
]

def load_testset():
    p = Path("data/medquad_test.jsonl")
    if p.exists():
        rows = []
        for line in p.read_text().splitlines():
            if not line.strip():
                continue
            r = json.loads(line)
            user = next(m["content"] for m in r["messages"] if m["role"] == "user")
            gold = json.loads(next(m["content"] for m in r["messages"] if m["role"] == "assistant"))
            rows.append({"question": user, "gold": gold})
        return rows
    return INLINE_TEST

TESTSET = load_testset()
print(f"Loaded {len(TESTSET)} test items (source: {'Week 4 JSONL' if Path('data/medquad_test.jsonl').exists() else 'inline'})")

---
## 1. Load a model to evaluate

We evaluate whatever you trained in Week 4 (`qwen3-medquad-*`), falling back to the **base** `Qwen/Qwen3-0.6B` so the notebook always runs. Evaluating the *base* model is itself useful: it's the **baseline** every fine-tune must beat.

In [ ]:
from unsloth import FastLanguageModel
from pathlib import Path
import torch

CANDIDATES = ["qwen3-medquad-qlora-merged", "qwen3-medquad-full-sft", "Qwen/Qwen3-0.6B"]
MODEL_PATH = next((c for c in CANDIDATES if c.startswith("Qwen/") or Path(c).exists()), "Qwen/Qwen3-0.6B")
print(f"Evaluating: {MODEL_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH, max_seq_length=1024, dtype=None, load_in_4bit=False)
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "You are a clinical assistant. Read the medical question and respond with a JSON object "
    "containing: disease, patient_info, symptoms (list), treatment (list), answer_summary.")

def predict(question, max_new_tokens=220):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True,
        enable_thinking=False,                       # Qwen3: answer directly (no <think>) so the JSON is parseable
        return_tensors="pt", return_dict=True).to(model.device)   # return_dict -> attention_mask passed too
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(predict(TESTSET[0]["question"])[:300])

---
## 2. The metrics

Three small, transparent functions. No black-box metric library where we can avoid it — for a teaching eval you should be able to read exactly what each number means.

- **`parse_json`** — extract the first JSON object from the model's text (models love to add prose around it). Returns `None` on failure -> counts as a JSON-validity miss.
- **`set_f1`** — precision/recall/F1 over two lists of strings, with light normalization (lowercase, fuzzy contains) so "shortness of breath" matches "breathlessness"-style near-misses only when one contains the other.
- **ROUGE-L** — standard longest-common-subsequence overlap for the free-text summary.

In [ ]:
import json, re
from rouge_score import rouge_scorer

_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def parse_json(text):
    """Grab the first {...} block and parse it. Returns dict or None."""
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def _norm(s):
    return re.sub(r"[^a-z0-9 ]", "", str(s).lower()).strip()

def set_f1(pred_list, gold_list):
    pred = {_norm(x) for x in (pred_list or []) if _norm(x)}
    gold = {_norm(x) for x in (gold_list or []) if _norm(x)}
    if not pred and not gold:
        return 1.0, 1.0, 1.0
    # a predicted item counts as a hit if it equals OR is contained-in/ contains a gold item
    def hit(p, S):
        return any(p == g or p in g or g in p for g in S)
    tp = sum(1 for p in pred if hit(p, gold))
    prec = tp / len(pred) if pred else 0.0
    rec  = sum(1 for g in gold if hit(g, pred)) / len(gold) if gold else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return prec, rec, f1

def disease_match(pred, gold):
    p, g = _norm(pred.get("disease", "")), _norm(gold.get("disease", ""))
    if not p:
        return 0.0
    return 1.0 if (p == g or p in g or g in p) else 0.0

def rouge_l(pred, gold_summary):
    s = pred.get("answer_summary", "")
    if not s or not gold_summary:
        return 0.0
    return _rouge.score(gold_summary, s)["rougeL"].fmeasure

---
## 3. Run the evaluation

For each test item: generate -> parse -> score every field. We aggregate into a single scorecard. (`gold` here has no `answer_summary` for the inline set, so ROUGE is reported only when a gold summary exists — honest about what we can and can't measure.)

In [ ]:
def evaluate(testset):
    rows, n_valid = [], 0
    for item in testset:
        raw = predict(item["question"])
        pred = parse_json(raw)
        if pred is None:
            rows.append({"json_ok": 0, "disease": 0.0, "symptom_f1": 0.0, "treatment_f1": 0.0})
            continue
        n_valid += 1
        gold = item["gold"]
        _, _, sf1 = set_f1(pred.get("symptoms"), gold.get("symptoms"))
        _, _, tf1 = set_f1(pred.get("treatment"), gold.get("treatment"))
        rows.append({"json_ok": 1, "disease": disease_match(pred, gold),
                     "symptom_f1": round(sf1, 3), "treatment_f1": round(tf1, 3)})
    def avg(k):
        return sum(r[k] for r in rows) / len(rows)
    return {"model": MODEL_PATH, "n": len(rows),
            "json_valid_rate": round(avg("json_ok"), 3),
            "disease_acc": round(avg("disease"), 3),
            "symptom_f1": round(avg("symptom_f1"), 3),
            "treatment_f1": round(avg("treatment_f1"), 3)}, rows

scorecard, detail = evaluate(TESTSET)
print(json.dumps(scorecard, indent=2))

---
## 4. Base vs fine-tuned — the comparison that matters

The single number that justifies a whole week of fine-tuning is **"how much better than the base model?"** If you trained models in Week 4, point `CANDIDATES` at each and rerun. Otherwise we tabulate your current run against **reference** scores so you can see the shape of the comparison.

> A common, humbling result: a 0.6B base model is *bad* at emitting clean JSON, and even a short fine-tune lifts JSON-validity and disease-accuracy sharply. That delta — not the absolute score — is the evidence that post-training worked.

In [ ]:
# Reference scorecards from representative runs (Qwen3-0.6B family) — fallback so the
# comparison is visible even if you only evaluated one model in this session.
REF = {
    "base (Qwen3-0.6B)":  {"json_valid_rate": 0.40, "disease_acc": 0.35, "symptom_f1": 0.22, "treatment_f1": 0.18},
    "full SFT":           {"json_valid_rate": 0.98, "disease_acc": 0.82, "symptom_f1": 0.61, "treatment_f1": 0.55},
    "LoRA":               {"json_valid_rate": 0.97, "disease_acc": 0.80, "symptom_f1": 0.58, "treatment_f1": 0.52},
    "QLoRA":              {"json_valid_rate": 0.96, "disease_acc": 0.78, "symptom_f1": 0.57, "treatment_f1": 0.50},
}
# Slot your live run in under its own label.
live_label = "YOUR RUN: " + scorecard["model"].split("/")[-1]
table = {live_label: {k: scorecard[k] for k in ["json_valid_rate", "disease_acc", "symptom_f1", "treatment_f1"]}}
table.update(REF)

cols = ["json_valid_rate", "disease_acc", "symptom_f1", "treatment_f1"]
hdr = f"{'model':24s} " + " ".join(f"{c:>14s}" for c in cols)
print(hdr); print("-" * len(hdr))
for name, sc in table.items():
    print(f"{name:24s} " + " ".join(f"{sc[c]:>14.3f}" for c in cols))
print("\nRead the BASE row vs the fine-tuned rows. That gap is the answer to 'did it work?'")

## What you learned

You evaluated a model the right way: held-out data, field-level metrics, against a baseline. You also saw the limits — a tiny model has a ceiling, and metrics only measure what you defined.

**Next — `01_llm_as_judge.ipynb`:** some qualities (helpfulness, faithfulness, tone) have no clean F1. For those we recruit a **model as the judge** — and learn exactly how much to trust it.